# Phase 6 — Forecasting

## Objective

The objective of this phase is to determine whether the available historical sales data is sufficient to support reliable demand forecasting and, where appropriate, generate simple and interpretable forecasts that can support inventory decisions.

This phase follows a baseline-first approach. Forecasting will only be used where the historical data contains enough information to justify it.

## What This Phase Will Achieve

By the end of this notebook, we aim to:

- Assess whether each product/category has enough historical periods for forecasting.
- Identify sparse, intermittent, or insufficient-demand series.
- Build a simple baseline forecast using historical sales demand.
- Evaluate forecasts using a chronological holdout rather than a random split.
- Compare any candidate model against the baseline.
- Assign each series a clear forecasting status:
  - Forecastable
  - Baseline Only
  - Insufficient History
- Produce a governed forecast output that can later support the Phase 5 Inventory Intelligence logic.

## Forecasting Principle

The goal is not to build the most complex model.

The goal is to build the simplest forecast that provides reliable business value.

If the available history is insufficient, the correct outcome of this phase may be a documented decision not to forecast that product or category.

## Expected Final Output

The final forecasting dataset will contain fields such as:

- Product_ID
- Model
- Category
- Forecast_Period
- Forecast_Units
- Forecast_Method
- Forecast_Error
- Forecast_Confidence
- Forecast_Status

This output will later be integrated with inventory recommendations to support more informed reorder and stock decisions.

In [4]:
# Load governed datasets
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent.parent

SALES_PATH = PROJECT_ROOT / "data" / "processed" / "fact_sales.csv"
fact_sales = pd.read_csv(SALES_PATH)

print("Shape:", fact_sales.shape)
display(fact_sales.head())

print("\nColumns:")
print(fact_sales.columns.tolist())

Shape: (966, 18)


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH



Columns:
['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']


### Step 1 — Forecast Readiness Check

Before building any forecasting model, the historical sales data is assessed to determine whether sufficient time-series information exists.

This step checks:

- available time periods
- number of observations per product
- missing or sparse demand history
- suitability for SKU-level forecasting

The purpose is to decide whether forecasting should proceed at product level, category level, or remain baseline-only.

In [5]:
print("Shape:", fact_sales.shape)

print("\nColumns:")
print(fact_sales.columns.tolist())

print("\nData Types:")
print(fact_sales.dtypes)

Shape: (966, 18)

Columns:
['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']

Data Types:
Sales_Record_ID                     int64
Product_ID                          int64
Product_Key                        object
Source_Month                       object
Category                           object
Stock Code                         object
Description                        object
Record_Type                        object
Level                               int64
Sold Period                         int64
Transaction_Status                 object
Unit Cost                         float64
Unit Price                        float64
Cost Sales                        float64
Sales Value                       float64
Profit                            float64
Profit % 

In [6]:
date_like_columns = [
    col for col in fact_sales.columns
    if any(x in col.lower() for x in ["date", "period", "month", "year"])
]

print("Possible time columns:")
print(date_like_columns)

Possible time columns:
['Source_Month', 'Sold Period']


In [11]:
month_order = {
    "Nov": 1,
    "Dec": 2,
    "Jan": 3
}

fact_sales["Month_Order"] = fact_sales["Source_Month"].map(month_order)

print("Unique source months:")
print(fact_sales["Source_Month"].unique())

print("\nMonth order check:")
display(
    fact_sales[["Source_Month", "Month_Order"]]
    .drop_duplicates()
    .sort_values("Month_Order")
)

Unique source months:
['Nov' 'Dec' 'Jan']

Month order check:


,Source_Month,Month_Order
0,Nov,1
381,Dec,2
704,Jan,3


In [12]:
print("Unique months:", fact_sales["Source_Month"].nunique())

periods_per_product = (
    fact_sales
    .groupby("Product_ID")["Source_Month"]
    .nunique()
    .reset_index(name="Available_Periods")
)

print("\nPeriods per product:")
display(
    periods_per_product["Available_Periods"]
    .value_counts()
    .sort_index()
)

Unique months: 3

Periods per product:


Available_Periods
1    602
2    138
3     27
Name: count, dtype: int64


The forecasting dataset is assessed to determine how many unique sales periods are available overall and per product.

This check establishes whether SKU-level forecasting is feasible or whether forecasting should be performed at a higher aggregation level such as category.

The governed sales dataset contains only three historical periods: November, December and January.

Product-level coverage is highly sparse:

- 602 products have only 1 observed period
- 138 products have 2 observed periods
- 27 products have all 3 observed periods

Therefore, SKU-level time-series forecasting is not considered sufficiently reliable.

Phase 6 will proceed using a simple aggregated baseline approach rather than complex forecasting models. The output will be treated as directional decision support rather than a production-grade demand forecast.

In [10]:
category_month_sales = (
    fact_sales
    .groupby(["Category", "Source_Month", "Month_Order"], as_index=False)
    ["Sold Period"]
    .sum()
    .sort_values(["Category", "Month_Order"])
)

category_coverage = (
    category_month_sales
    .groupby("Category")["Source_Month"]
    .nunique()
    .reset_index(name="Available_Periods")
)

print("Category period coverage:")
display(
    category_coverage["Available_Periods"]
    .value_counts()
    .sort_index()
)

display(category_month_sales.head(15))

Category period coverage:


Available_Periods
1    14
2    20
3    54
Name: count, dtype: int64

,Category,Source_Month,Month_Order,Sold Period
2,ACCESSORIES,Nov,1,15
0,ACCESSORIES,Dec,2,13
1,ACCESSORIES,Jan,3,9
3,BAGS,Nov,1,1
6,BLENDERS,Nov,1,5
4,BLENDERS,Dec,2,7
5,BLENDERS,Jan,3,6
7,BUILT UNDER DBL OVEN,Dec,2,1
10,BULBS,Nov,1,7
8,BULBS,Dec,2,3


Category-level aggregation improves historical coverage compared with SKU-level data.

Out of 88 categories:

- 54 categories contain all 3 available months
- 20 categories contain 2 months
- 14 categories contain only 1 month

Although 54 categories have complete coverage, three monthly observations are still insufficient for robust time-series modelling.

Therefore:

- Forecasting will be performed at **category level**
- Only categories with all 3 months will be included
- A simple baseline forecasting method will be used
- Complex forecasting models are intentionally excluded due to insufficient historical depth

The forecast should be interpreted as directional demand guidance rather than a statistically mature production forecast.

### Step 2 - Create Forecast-Ready Datase

In [13]:
## 
forecastable_categories = category_coverage.loc[
    category_coverage["Available_Periods"] == 3,
    "Category"
]

forecast_ready_df = (
    category_month_sales[
        category_month_sales["Category"].isin(forecastable_categories)
    ]
    .sort_values(["Category", "Month_Order"])
    .reset_index(drop=True)
)

print("Forecastable categories:", forecast_ready_df["Category"].nunique())
print("Rows:", len(forecast_ready_df))

display(forecast_ready_df.head(15))

Forecastable categories: 54
Rows: 162


,Category,Source_Month,Month_Order,Sold Period
0,ACCESSORIES,Nov,1,15
1,ACCESSORIES,Dec,2,13
2,ACCESSORIES,Jan,3,9
3,BLENDERS,Nov,1,5
4,BLENDERS,Dec,2,7
5,BLENDERS,Jan,3,6
6,BULBS,Nov,1,7
7,BULBS,Dec,2,3
8,BULBS,Jan,3,6
9,CABLES,Nov,1,14


### Step 3 — Baseline Forecast

A simple moving-average baseline is used because only three historical monthly observations are available per category.

The forecast for the next period is calculated as the average demand across the available three months.

This provides a transparent baseline that can support directional planning without overstating forecast reliability.

In [14]:
baseline_forecast = (
    forecast_ready_df
    .groupby("Category", as_index=False)
    .agg(
        Historical_Avg_Demand=("Sold Period", "mean"),
        Min_Demand=("Sold Period", "min"),
        Max_Demand=("Sold Period", "max")
    )
)

baseline_forecast["Forecast_Units"] = (
    baseline_forecast["Historical_Avg_Demand"]
    .round()
    .astype(int)
)

baseline_forecast["Forecast_Method"] = "3-Month Moving Average"

display(baseline_forecast.head(15))

,Category,Historical_Avg_Demand,Min_Demand,Max_Demand,Forecast_Units,Forecast_Method
0,ACCESSORIES,12.333333,9,15,12,3-Month Moving Average
1,BLENDERS,6.000000,5,7,6,3-Month Moving Average
2,BULBS,5.333333,3,7,5,3-Month Moving Average
3,CABLES,13.666667,9,18,14,3-Month Moving Average
4,COFFEE ACCESSORIES,2.000000,2,2,2,3-Month Moving Average
5,COFFEE MAKERS,9.333333,7,11,9,3-Month Moving Average
6,COOKERS,4.000000,2,6,4,3-Month Moving Average
7,CYLINDER VACS,3.666667,3,5,4,3-Month Moving Average
8,DELIVERY CHARGE,7.333333,2,12,7,3-Month Moving Average
9,DOUBLE OVENS,2.333333,1,5,2,3-Month Moving Average


### Step 4 — Baseline Validation

To evaluate the baseline forecast without random splitting, November and December are used as the historical training period and January is used as the holdout period.

The forecast for January is calculated as the average demand from November and December.

Forecast accuracy is evaluated using MAE and WAPE.

In [15]:
# Training data: Nov + Dec
train_df = forecast_ready_df[
    forecast_ready_df["Source_Month"].isin(["Nov", "Dec"])
]

# Actual holdout: Jan
jan_actual = (
    forecast_ready_df[
        forecast_ready_df["Source_Month"] == "Jan"
    ][["Category", "Sold Period"]]
    .rename(columns={"Sold Period": "Actual_Jan"})
)

# Baseline prediction from Nov + Dec average
jan_forecast = (
    train_df
    .groupby("Category", as_index=False)
    .agg(Forecast_Jan=("Sold Period", "mean"))
)

validation_df = jan_forecast.merge(
    jan_actual,
    on="Category",
    how="inner"
)

validation_df["Absolute_Error"] = (
    validation_df["Actual_Jan"] -
    validation_df["Forecast_Jan"]
).abs()

display(validation_df.head(15))

,Category,Forecast_Jan,Actual_Jan,Absolute_Error
0,ACCESSORIES,14.0,9,5.0
1,BLENDERS,6.0,6,0.0
2,BULBS,5.0,6,1.0
3,CABLES,16.0,9,7.0
4,COFFEE ACCESSORIES,2.0,2,0.0
5,COFFEE MAKERS,9.0,10,1.0
6,COOKERS,5.0,2,3.0
7,CYLINDER VACS,3.0,5,2.0
8,DELIVERY CHARGE,7.0,8,1.0
9,DOUBLE OVENS,3.0,1,2.0


In [16]:
mae = validation_df["Absolute_Error"].mean()

wape = (
    validation_df["Absolute_Error"].sum()
    / validation_df["Actual_Jan"].sum()
) * 100

print(f"MAE: {mae:.2f} units")
print(f"WAPE: {wape:.2f}%")

MAE: 2.77 units
WAPE: 50.51%


The 3-month moving-average baseline produced:

- MAE: 2.77 units
- WAPE: 50.51%

The error level is high, indicating that the available three-month history does not provide sufficient stability for reliable category-level demand forecasting.

Therefore, the forecast should not be used as a production demand forecast or as an automatic reorder trigger.

One additional simple naive baseline will be tested before the final Phase 6 decision.

In [17]:
# Naive forecast: December demand predicts January demand

dec_forecast = (
    forecast_ready_df[
        forecast_ready_df["Source_Month"] == "Dec"
    ][["Category", "Sold Period"]]
    .rename(columns={"Sold Period": "Forecast_Jan"})
)

naive_validation = dec_forecast.merge(
    jan_actual,
    on="Category",
    how="inner"
)

naive_validation["Absolute_Error"] = (
    naive_validation["Actual_Jan"] -
    naive_validation["Forecast_Jan"]
).abs()

naive_mae = naive_validation["Absolute_Error"].mean()

naive_wape = (
    naive_validation["Absolute_Error"].sum()
    / naive_validation["Actual_Jan"].sum()
) * 100

print(f"Naive MAE: {naive_mae:.2f} units")
print(f"Naive WAPE: {naive_wape:.2f}%")

Naive MAE: 3.00 units
Naive WAPE: 54.73%


Two simple baseline methods were evaluated using January as a time-based holdout.

| Method | MAE | WAPE |
|---|---:|---:|
| 3-Month Moving Average | 2.77 | 50.51% |
| Naive Forecast | 3.00 | 54.73% |

The 3-month moving-average baseline performs slightly better than the naive approach.

However, both methods show high forecast error. Combined with only three months of historical data, this indicates that the current dataset is insufficient for reliable production forecasting.

### Decision

- Complex forecasting models will not be developed.
- The 3-month moving average will be retained only as a **directional baseline**.
- Forecast outputs must not automatically drive reorder decisions.
- Phase 6 will be classified as **Baseline-Only / Insufficient Historical Depth**.
- More historical monthly data should be collected before advanced forecasting is reconsidered.

In [18]:
final_forecast = baseline_forecast[
    [
        "Category",
        "Historical_Avg_Demand",
        "Forecast_Units",
        "Forecast_Method"
    ]
].copy()

final_forecast["Forecast_Error_WAPE"] = 50.51
final_forecast["Forecast_Confidence"] = "Low"
final_forecast["Forecast_Status"] = "Baseline Only"

display(final_forecast.head(15))

,Category,Historical_Avg_Demand,Forecast_Units,Forecast_Method,Forecast_Error_WAPE,Forecast_Confidence,Forecast_Status
0,ACCESSORIES,12.333333,12,3-Month Moving Average,50.51,Low,Baseline Only
1,BLENDERS,6.000000,6,3-Month Moving Average,50.51,Low,Baseline Only
2,BULBS,5.333333,5,3-Month Moving Average,50.51,Low,Baseline Only
3,CABLES,13.666667,14,3-Month Moving Average,50.51,Low,Baseline Only
4,COFFEE ACCESSORIES,2.000000,2,3-Month Moving Average,50.51,Low,Baseline Only
5,COFFEE MAKERS,9.333333,9,3-Month Moving Average,50.51,Low,Baseline Only
6,COOKERS,4.000000,4,3-Month Moving Average,50.51,Low,Baseline Only
7,CYLINDER VACS,3.666667,4,3-Month Moving Average,50.51,Low,Baseline Only
8,DELIVERY CHARGE,7.333333,7,3-Month Moving Average,50.51,Low,Baseline Only
9,DOUBLE OVENS,2.333333,2,3-Month Moving Average,50.51,Low,Baseline Only


The available sales history is insufficient for reliable production-grade time-series forecasting.

Only three monthly periods are available, and SKU-level historical coverage is highly sparse. Category-level aggregation improved coverage, allowing 54 categories to be evaluated across all three months.

Two baseline methods were tested using January as a chronological holdout:

| Method | MAE | WAPE |
|---|---:|---:|
| 3-Month Moving Average | 2.77 | 50.51% |
| Naive Forecast | 3.00 | 54.73% |

The 3-month moving average performed slightly better and was retained as the directional baseline.

However, its high validation error means the forecasts should not be used as automatic reorder triggers or treated as production demand forecasts.

### Phase 6 Decision

**Status: Baseline Only — Low Confidence**

Advanced models such as ARIMA, SARIMA, Prophet or machine-learning forecasting are intentionally not implemented because the historical depth is insufficient to support reliable modelling.

More historical monthly sales data should be accumulated before advanced forecasting is reconsidered.